# Math 05: Linear Algebra — Vectors**Learning Objectives:**1. Represent points and displacements in N-dimensional space2. Perform vector addition and scalar multiplication3. Compute the dot product and understand its geometric meaning4. Compute the cross product (for 3D vectors)5. Implement vector math computationally from scratch**Prerequisites:** Math 01 (Algebra)**Estimated time:** 90 minutes**Textbook:** *Justin Skycak — Linear Algebra* §1.1-1.2

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "math_05_linear_algebra_vectors",    "level": 1,    "title": "Math 05: Linear Algebra \u2014 Vectors",    "prerequisites": [        "Math 01 (Algebra)"    ],    "skills_taught": [        "lesson.math_05_linear_algebra_vectors"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "math_06_linear_algebra_matrices.ipynb"}SKILLS = [    {        "id": "lesson.math_05_linear_algebra_vectors",        "title": "Lesson Math 05 Linear Algebra Vectors",        "notebook": "math_05_linear_algebra_vectors.ipynb",        "level": 1    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "math_05_linear_algebra_vectors.ipynb",        "level": 1    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Math 01 (Algebra).

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.math_05_linear_algebra_vectors', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setup!pip install -q sympy matplotlib plotly ipywidgetsimport sympy as sp; sp.init_printing()import numpy as np, matplotlib.pyplot as pltprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### What is a Vector?> 📖 *From Linear Algebra §1.1:* Points in N-dimensional space consist of numbers, but can also be thought of as manipulable entities in their own right, called **vectors**. Instead of fixed points, we think of them as displacements through space.A vector $\vec{v} = \langle v_1, v_2, \dots, v_n \rangle$ represents a displacement.### Vector Algebra1. **Addition (Component-wise):** $\langle 1, 2 \rangle + \langle 3, 4 \rangle = \langle 4, 6 \rangle$   - *Geometry:* Start the second vector where the first one ends.2. **Scalar Multiplication:** $3 \cdot \langle 1, -2 \rangle = \langle 3, -6 \rangle$   - *Geometry:* Rescales the vector's length. Negative scalars flip its direction.3. **Norm (Length):** $||\vec{v}|| = \sqrt{v_1^2 + v_2^2 + \dots + v_n^2}$ (Pythagorean theorem)

### Multiplication of Vectors> 📖 *From Linear Algebra §1.2:* What does it mean to multiply a vector by another vector? The two most common interpretations are the **dot product** and the **cross product**.**1. Dot Product (Scalar Output)**$\vec{a} \cdot \vec{b} = a_1 b_1 + a_2 b_2 + \dots + a_n b_n$*Geometry:* $\vec{a} \cdot \vec{b} = ||\vec{a}|| ||\vec{b}|| \cos(\theta)$, where $\theta$ is the angle between them.- If $\vec{a} \cdot \vec{b} = 0$, the vectors are **perpendicular** (orthogonal).**2. Cross Product (Vector Output, 3D only)**$\vec{a} \times \vec{b} = \langle a_2 b_3 - a_3 b_2,\; a_3 b_1 - a_1 b_3,\; a_1 b_2 - a_2 b_1 \rangle$*Geometry:* Produces a new vector **perpendicular to both** $\vec{a}$ and $\vec{b}$.Its length $||\vec{a} \times \vec{b}|| = ||\vec{a}|| ||\vec{b}|| \sin(\theta)$ is the area of the parallelogram formed by $\vec{a}$ and $\vec{b}$.

In [ ]:
# Symbolic verification of orthogonalityimport sympy as spa1, a2, a3, b1, b2, b3 = sp.symbols('a1 a2 a3 b1 b2 b3')a = sp.Matrix([a1, a2, a3])b = sp.Matrix([b1, b2, b3])cross_prod = a.cross(b)print('Cross product:\n', cross_prod)print('\nDot product of a with (a x b) should be 0:')print('Result:', sp.simplify(a.dot(cross_prod)))

### Comprehension Check ✅1. What is $\langle 1, 0, 0 \rangle \cdot \langle 0, 1, 0 \rangle$?2. What is the length (norm) of $\langle 3, 4 \rangle$?3. If scalar multiplication by $-1$ flips a vector, what is $\vec{a} + (-\vec{a})$?<details><summary>💡 Explanation after a retrieval attempt</summary>1. 0 (they are perpendicular axes: x and y).2. $\sqrt{3^2 + 4^2} = \sqrt{25} = 5$.3. $\langle 0, 0, \dots, 0 \rangle$ (the zero vector).</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: Compute by HandGiven $\vec{u} = \langle 1, -2, 3 \rangle$ and $\vec{v} = \langle 0, 4, 1 \rangle$:1. Compute $\vec{u} + \vec{v}$2. Compute $2\vec{u} - \vec{v}$3. Compute $\vec{u} \cdot \vec{v}$Do the math on paper, then write code to verify using standard Python lists.

In [ ]:
def vector_math_practice():    u = [1, -2, 3]    v = [0, 4, 1]    # YOUR CODE HERE to compute the 3 results    pass# Note: You can't just do u + v in pure Python, because that concatenates lists!# [1,-2,3] + [0,4,1] = [1,-2,3,0,4,1]. You must add element-by-element.

---## Phase 1 — Algorithm ConstructionNow we will build a basic linear algebra library from scratch using Python lists (no numpy).### Step 1: Addition and Scalar Multiplication

In [ ]:
def vec_add(u, v):    '''Add two vectors of the same dimension.'''    # YOUR CODE HERE    passdef vec_scale(scalar, v):    '''Multiply a vector by a scalar.'''    # YOUR CODE HERE    pass# Tests:# assert vec_add([1, 2], [3, 4]) == [4, 6]# assert vec_scale(3, [1, -2]) == [3, -6]# print('Step 1 passed ✅')

### Step 2: Dot Product and Norm

In [ ]:
import mathdef dot_product(u, v):    '''Compute the dot product of two vectors.'''    # YOUR CODE HERE    passdef norm(v):    '''Compute the length of a vector using the dot product.'''    # YOUR CODE HERE    pass# Tests:# assert dot_product([1, 3, -5], [4, -2, -1]) == 3# assert norm([3, 4]) == 5.0# print('Step 2 passed ✅')

### Step 3: Finding the Angle (Interleaved Practice)Use the formula $\vec{a} \cdot \vec{b} = ||\vec{a}|| ||\vec{b}|| \cos(\theta)$ to find the angle between two vectors in radians.

In [ ]:
def angle_between(u, v):    '''Return the angle between u and v in radians.'''    import math    # YOUR CODE HERE    pass# Test:# angle = angle_between([1, 0], [0, 1])# assert abs(angle - math.pi/2) < 1e-5  # 90 degrees# print('Step 3 passed ✅')

---## Phase 2 — Experimental Verification### Floating Point and OrthogonalityWhen computing angles or dot products computationally, you rarely get exactly `0.0`. Let's visualize how the dot product behaves as we rotate a vector.

In [ ]:
import numpy as np, matplotlib.pyplot as plt# Vector u is fixed on the x-axisu = np.array([1.0, 0.0])# Vector v rotates from 0 to 2*pithetas = np.linspace(0, 2*np.pi, 100)dot_products = []for theta in thetas:    v = np.array([np.cos(theta), np.sin(theta)])    dot_products.append(np.dot(u, v))plt.figure(figsize=(8,4))plt.plot(thetas, dot_products, 'b-', lw=2)plt.axhline(0, color='r', ls='--')plt.xlabel('Angle θ (radians)')plt.ylabel('Dot Product (u • v)')plt.title('Dot Product vs Angle for Unit Vectors')plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Pathological Case: Catastrophic Cancellation in AnglesWhen vectors are very close to parallel (angle $\approx 0$), computing the angle via $\cos^{-1}$ can fail due to precision loss. Specifically, if floating point math makes $\frac{\vec{a}\cdot\vec{b}}{||a|| ||b||}$ slightly greater than 1, `math.acos` will throw a Domain Error!

In [ ]:
import matha = [1.0, 2.0, 3.0]b = [1.000000000000001, 2.000000000000002, 3.000000000000003] # Almost identical# Try to compute angle directly:# val = dot_product(a, b) / (norm(a) * norm(b))# print('Value passed to acos:', val)# angle = math.acos(val) # May crash!

---## Phase 3 — Olympiad Extension### Challenge: N-Dimensional Cross Product (Exterior Product)The standard cross product only works in 3D. But there is a mathematical generalization called the **Wedge Product** (or exterior product) that works in any dimension. Alternatively, in N-dimensions, given $N-1$ vectors, you can find a vector orthogonal to all of them using a generalized cross product (computed via a matrix determinant).Write a function that takes two 3D vectors and computes their standard cross product, WITHOUT using numpy.

In [ ]:
def cross_product_3d(u, v):    '''Return the 3D cross product of u and v.'''    # YOUR CODE HERE    pass# Test:# assert cross_product_3d([1,0,0], [0,1,0]) == [0,0,1]

### Chalkboard DefenseWrite a formal mathematical defense of why `a.dot(a x b) = 0`. Use the algebraic formula for the cross product to expand the terms and show they cancel out exactly.*(Write your defense below)*<details><summary>💡 Sketch after a retrieval attempt</summary>Let $\vec{c} = \vec{a} \times \vec{b} = \langle a_2b_3 - a_3b_2,\; a_3b_1 - a_1b_3,\; a_1b_2 - a_2b_1 \rangle$.$\vec{a} \cdot \vec{c} = a_1(a_2b_3 - a_3b_2) + a_2(a_3b_1 - a_1b_3) + a_3(a_1b_2 - a_2b_1)$$= a_1a_2b_3 - a_1a_3b_2 + a_2a_3b_1 - a_1a_2b_3 + a_1a_3b_2 - a_2a_3b_1 = 0$.</details>---📚 **Next:** Math 06 (Linear Algebra — Matrices)

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.math_05_linear_algebra_vectors', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='math_06_linear_algebra_matrices.ipynb')